# PageRank y HITS (IV-H) + índice de influencia I(h)
PageRank sobre la proyección habilidad–habilidad (PMI), α=0.85 con sensibilidad. HITS sobre aristas dirigidas oferta→habilidad.

In [3]:
import pickle
import pandas as pd, networkx as nx

with open("grafo.pkl", "rb") as f: G = pickle.load(f)
skills = [n for n, d in G.nodes(data=True) if d["tipo"] == "habilidad"]

# Proyección de habilidades ponderada por PMI
Gs = nx.Graph()
Gs.add_nodes_from(skills)
Gs.add_weighted_edges_from((u, v, d["weight"]) for u, v, d in G.edges(data=True) if d["rel"] == "coocurre")

# PageRank y sensibilidad de alfa
pr = {a: pd.Series(nx.pagerank(Gs, alpha=a, weight="weight")) for a in (0.70, 0.85, 0.95)}
print("Spearman PR(0.85) vs 0.70 / 0.95:",
      round(pr[0.85].corr(pr[0.70], method="spearman"), 3),
      round(pr[0.85].corr(pr[0.95], method="spearman"), 3))

Spearman PR(0.85) vs 0.70 / 0.95: 0.995 0.993


In [4]:
# HITS sobre oferta -> habilidad (autoridad = habilidad requerida por ofertas "hub")
Gd = nx.DiGraph((u, v) for u, v, d in G.edges(data=True) if d["rel"] == "requiere")
hubs, auth = nx.hits(Gd, max_iter=1000, normalized=True)

res = pd.DataFrame({"habilidad": skills})
res["pagerank"] = res.habilidad.map(pr[0.85])
res["authority"] = res.habilidad.map(auth).fillna(0)
res["grado_ofertas"] = res.habilidad.map(lambda h: Gd.in_degree(h) if h in Gd else 0)

# Índice de influencia estructural: I(h) = (percentil(PR) + percentil(Auth)) / 2
res["I"] = (res.pagerank.rank(pct=True) + res.authority.rank(pct=True)) / 2
res = res.sort_values("I", ascending=False).reset_index(drop=True)
res.to_csv("skill_influencia.csv", index=False)
res.head(20)

,habilidad,pagerank,authority,grado_ofertas,I
0,ingles,0.027080,0.046950,109,0.888060
1,excel,0.017025,0.113728,221,0.865672
2,power bi,0.028672,0.033666,70,0.865672
3,kpi,0.019178,0.060787,128,0.858209
4,sql,0.060716,0.011895,31,0.843284
5,gestión de proyectos,0.019656,0.030808,57,0.820896
6,agile,0.053027,0.009904,24,0.813433
7,costos y presupuestos,0.015813,0.062940,122,0.805970
8,comunicación,0.015352,0.068107,143,0.791045
9,api rest,0.054353,0.004307,13,0.783582
